# 03 — M4: schedule-level Monte Carlo (research only)

A win total is a **distribution** question. M1–M3 and M5 emit a point estimate; M4 simulates the
actual schedule and reports the whole win distribution per team.

**How it works.** Fit a team rating in margin units from prior-information features, plus a
home-field constant and a residual spread — all on training seasons only. Then, for each scheduled
game of season *T*, draw a margin, decide the game, and repeat 20,000 times. A team's win
distribution is what falls out.

**Why simulate rather than regress:**

* **League wins are conserved by construction** — every simulated game awards exactly 1.0 across the
  two teams, so the 32 projections always sum to the games played. No regression gives that free.
* **Opponent quality enters through the real schedule**, and outcomes are correlated the way they
  actually are (both teams cannot win).
* It produces intervals, which is what a projections surface actually needs.

**Venue (A2.5.6).** Home-field is removed for **explicit-neutral or international** games only. A
domestic alternate venue keeps normal home field — zeroing it would need its own preregistered rule.

**Amendment 3 (2026-08-03).** The first run measured M4's 80% interval covering only **65%** —
over-confident. A3 declares **M4-c**: the same model plus a per-team-season strength shock
`epsilon ~ N(0, tau^2)`, with `tau` selected **on inner training folds only**. Both models are
reported side by side; the original M4 numbers are not overwritten.

**Scope under `GO-TIER-B`.** CRPS, interval coverage, PIT and MAE are computed. **P(OVER) against
the posted number, its log loss / Brier, and push settlement are NOT computed** — all are gate-C
material and gate C is shut.

```bash
papermill futures/season_team_totals/03_distribution_model.ipynb /tmp/out.ipynb
```

## Section 1 — Parameters

Paths, the simulation size and the seed. Folds, features and the venue rule are read from the frozen
artifacts.

In [ ]:
AUDIT_PATH   = None      # None -> futures/artifacts/data_audit.json
PANEL_PATH   = None      # None -> futures/data/team_season_panel.parquet
META_PATH    = None      # None -> futures/artifacts/dataset_metadata.json
VENUE_PATH   = None      # None -> futures/data/season_schedule_context.parquet
SCHED_PATH   = None      # None -> futures/data/schedules_snapshot.parquet
OUT_PATH     = None      # None -> futures/artifacts/distribution_eval.json
N_SIMS       = 20000
SEED         = 20260802
WRITE_ARTIFACTS = True
RUN_TESTS    = True

### Interpreting the output

Silent. `N_SIMS` is the only modelling knob here, and it controls Monte Carlo noise — not the model.
20,000 draws puts the standard error of a mean win count near 0.02 wins, well below anything the
metrics resolve.

### What these tests guard

That the design is not injectable, and that the simulation is large enough for its own noise to be
negligible against the effects being measured.

In [ ]:
if RUN_TESTS:
    assert isinstance(SEED, int) and N_SIMS >= 10000, "too few draws for stable win distributions"
    for _n in ("FOLDS", "HFA", "SIGMA", "RATING"):
        assert _n not in dir(), f"{_n} must be fitted or read, never a parameter"
    print(f"✓ Section 1 tests passed | {N_SIMS:,} simulations, seed {SEED}")

### Reading the test result

Confirms the knobs are the simulation size and the seed. Does **not** prove 20,000 is enough — the
determinism and conservation checks in Section 5 speak to that.

## Section 2 — Gate, load, and the venue rule

Loads the audit, the panel, the schedule snapshot and the venue authority `01` wrote. Builds the
per-game home-field multiplier from A2.5.6: **0 for explicit-neutral or international, 1 otherwise.**

In [ ]:
import hashlib
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start}")


REPO    = _find_repo_root(Path.cwd())
FUTURES = REPO / "futures"
ART     = FUTURES / "artifacts"
DATA    = FUTURES / "data"
AUDIT = Path(AUDIT_PATH) if AUDIT_PATH else ART / "data_audit.json"
PANEL = Path(PANEL_PATH) if PANEL_PATH else DATA / "team_season_panel.parquet"
META  = Path(META_PATH) if META_PATH else ART / "dataset_metadata.json"
VENUE = Path(VENUE_PATH) if VENUE_PATH else DATA / "season_schedule_context.parquet"
SCHED = Path(SCHED_PATH) if SCHED_PATH else DATA / "schedules_snapshot.parquet"
OUT   = Path(OUT_PATH) if OUT_PATH else ART / "distribution_eval.json"


def _rel(p):
    p = Path(p)
    try:
        return p.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(p.resolve())


def sha256_frame(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df.reset_index(drop=True),
                                                     index=False).values.tobytes()).hexdigest()


audit = json.loads(AUDIT.read_text(encoding="utf-8"))
meta = json.loads(META.read_text(encoding="utf-8"))
VERDICT = audit["verdict"]
if not VERDICT.startswith("GO"):
    raise RuntimeError(f"audit verdict is {VERDICT} — 03 must not run")
TIER_C_OPEN = bool(audit.get("tier_c_open", False))
FOLDS        = list(audit["folds"]["test_seasons"])
FOLDS_STRICT = list(audit["folds_strict_sensitivity"]["test_seasons"])
TARGET       = audit["target"]["column"]
FEATURES     = list(meta["features"]["columns"])

panel = pd.read_parquet(PANEL)
PANEL_HASH = sha256_frame(panel[meta["panel"]["columns"]])
venue = pd.read_parquet(VENUE)
# window the schedule exactly as 01 did, else the venue join is not 1:1
SEASON_MIN_AUDIT = int(audit["outcomes"]["season_min"])
SEASON_MAX_AUDIT = max(int(audit["outcomes"]["season_max"]), int(audit["predict_season"]["season"]))
sched = pd.read_parquet(SCHED)
sched = sched[(sched["game_type"] == "REG") &
              sched["season"].between(SEASON_MIN_AUDIT, SEASON_MAX_AUDIT)].copy()
FR = {"OAK": "LV", "SD": "LAC", "STL": "LA"}
sched["home_franchise"] = sched["home_team"].replace(FR)
sched["away_franchise"] = sched["away_team"].replace(FR)

# A2.5.6 home-field rule
venue["no_home_field"] = venue["explicit_neutral"] | venue["international_game"]
games = sched.merge(venue[["game_id", "no_home_field", "explicit_neutral", "international_game",
                           "non_primary_home_venue"]], on="game_id", how="inner")
games["hfa_mult"] = np.where(games["no_home_field"], 0.0, 1.0)

sys.path.insert(0, str(FUTURES / "season_team_totals"))
from tier_lock import TierCViolation, assert_no_tier_c

ALLOWED = {VERDICT, audit.get("tier_available", ""), "prior_off_epa_play", "prior_def_epa_play"}


def guard(obj, where):
    assert_no_tier_c(obj, where, allowed_literals=ALLOWED, tier_c_open=TIER_C_OPEN)


RUN_AT = datetime.now(timezone.utc)
PROVENANCE = {"notebook": "futures/season_team_totals/03_distribution_model.ipynb",
              "run_at_utc": RUN_AT.isoformat(), "python": sys.version.split()[0],
              "platform": platform.platform(), "pandas": pd.__version__, "numpy": np.__version__,
              "seed": SEED, "n_sims": int(N_SIMS), "audit_verdict": VERDICT}

_dom_alt = int((games["non_primary_home_venue"] & ~games["no_home_field"]).sum())
print(f"verdict     : {VERDICT}  (gate C open: {TIER_C_OPEN})")
print(f"panel       : {len(panel):,} rows, hash matches metadata: {PANEL_HASH == meta['panel']['frame_sha256']}")
print(f"games       : {len(games):,} REG games with venue context")
print(f"home field removed (A2.5.6): {int(games['no_home_field'].sum())} games "
      f"(explicit-neutral or international)")
print(f"domestic alternate venues KEEPING home field: {_dom_alt} games")

### Interpreting the output

Home field is removed from **84** games — every explicitly neutral or international one. The
**7** domestic-alternate games (New Orleans 2005, Minnesota 2010) **keep** normal home field,
because A2.5.6 forbids zeroing them without a rule of their own. That is a deliberate, visible
choice rather than a silent default.

### What these tests guard

The gate, the panel identity, and that the venue rule is applied exactly as A2.5.6 states — no game
loses home field for being at a domestic alternate venue alone.

In [ ]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B") and TIER_C_OPEN is False
    assert PANEL_HASH == meta["panel"]["frame_sha256"], "panel differs from the one 01 wrote"
    assert len(games) == len(sched), "the venue join dropped or duplicated games"
    # A2.5.6, both directions
    assert (games.loc[games["explicit_neutral"] | games["international_game"], "hfa_mult"] == 0).all()
    assert (games.loc[~(games["explicit_neutral"] | games["international_game"]), "hfa_mult"] == 1).all()
    _bad = games[(games["non_primary_home_venue"]) & (~games["no_home_field"]) & (games["hfa_mult"] != 1)]
    assert _bad.empty, "a domestic alternate venue lost home field without a preregistered rule"
    try:
        guard({"note": "a betting edge"}, "selftest")
        raise AssertionError("guard inactive")
    except TierCViolation:
        pass
    print(f"✓ Section 2 tests passed | {len(games):,} games, home field removed on "
          f"{int(games['no_home_field'].sum())} (neutral/international only), "
          f"{_dom_alt} domestic alternates keep it, guard live")

### Reading the test result

The A2.5.6 rule is enforced in both directions. Does **not** claim the rule is *right* — whether a
displaced domestic host really retains home field is an open empirical question the preregistration
deliberately declined to answer by fiat.

## Section 3 — Game-level training frame

M4 needs something M1–M3 never did: a **game-level** model. Each training game is joined to both
teams' season features, and the model target is the home margin.

The feature vector for a game is the **difference** `x_home − x_away`, so a rating falls out
naturally: a team's rating is the same linear function applied to its own features.

In [ ]:
feat = panel.set_index(["season", "franchise"])[FEATURES]


def game_frame(seasons):
    g = games[games["season"].isin(seasons) & games["result"].notna()].copy()
    h = feat.reindex(pd.MultiIndex.from_arrays([g["season"], g["home_franchise"]]))
    a = feat.reindex(pd.MultiIndex.from_arrays([g["season"], g["away_franchise"]]))
    X = pd.DataFrame(h.to_numpy() - a.to_numpy(), columns=FEATURES, index=g.index)
    return g, X


def predict_frame(season):
    g = games[games["season"] == season].copy()
    h = feat.reindex(pd.MultiIndex.from_arrays([g["season"], g["home_franchise"]]))
    a = feat.reindex(pd.MultiIndex.from_arrays([g["season"], g["away_franchise"]]))
    return g, pd.DataFrame(h.to_numpy() - a.to_numpy(), columns=FEATURES, index=g.index)


_g, _X = game_frame([s for s in FOLDS if s < FOLDS[-1]][:1] or [FOLDS[0] - 1])
print(f"example training frame: {len(_g):,} games x {_X.shape[1]} differenced features")
print(f"margin (home) mean {_g['result'].mean():+.3f}  std {_g['result'].std():.3f}")
print(f"raw home win rate in that sample: {(_g['result'] > 0).mean():.3f}")
print(f"tie rate over all settled games : {(games['result'] == 0).mean():.5f}")

### Interpreting the output

The differenced design is why this works: `x_home − x_away` is antisymmetric, so swapping the teams
flips the predicted margin and the model cannot learn a spurious "home teams are better" effect
through the features. Home advantage is carried by its own intercept, fitted separately.

The mean home margin is positive — that intercept is real and worth about two to three points
historically. Ties run about **0.2%** of games, rare but not zero, and §2.1 grades them at half a win.

### What these tests guard

The join is complete (no game silently loses a team's features), the design is genuinely
antisymmetric, and the margin column is the home-team margin rather than an absolute value.

In [ ]:
if RUN_TESTS:
    _gg, _XX = game_frame([2014])
    assert len(_gg) == 256, "unexpected 2014 game count"
    # the JOIN must have matched every game (a feature that is never null proves the key hit);
    # genuine NaNs remain in e.g. coach_prior_win_pct (<16 career games) and are median-imputed
    assert _XX["games_scheduled"].notna().all(), "the team-season join missed a 2014 game"
    assert _XX.isna().mean().mean() < 0.10, "unexpectedly many missing feature values"
    # antisymmetry: swapping home/away must negate every differenced feature
    _swapped = feat.reindex(pd.MultiIndex.from_arrays([_gg["season"], _gg["away_franchise"]])).to_numpy() \
        - feat.reindex(pd.MultiIndex.from_arrays([_gg["season"], _gg["home_franchise"]])).to_numpy()
    assert np.allclose(_swapped, -_XX.to_numpy(), equal_nan=True), "design not antisymmetric"
    assert (_gg["result"] == _gg["home_score"] - _gg["away_score"]).all(), "result is not the home margin"
    assert _gg["result"].mean() > 0, "no home advantage in the raw margins at all?"
    print(f"✓ Section 3 tests passed | game frames complete, design antisymmetric, "
          f"result == home margin, tie rate {(games['result'] == 0).mean():.5f}")

### Reading the test result

The design is antisymmetric, so home advantage can only enter through its own term. Does **not**
verify the *rating* is meaningful — that is the fit.

## Section 4 — Fit rating, home field and residual spread (training only)

Ridge on the differenced features predicts home margin, with an intercept that **is** the home-field
constant — because on a neutral game the differenced design still applies but the intercept must not.
So home field is fitted as an explicit column (`hfa_mult`) rather than as the model intercept, which
lets it switch off per A2.5.6.

`σ` is the residual standard deviation on training games. The tie threshold is calibrated so the
simulated tie rate matches the training tie rate.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

ALPHA_GRID = tuple(meta["m5_contract"]["alpha_grid"])
FALLBACK_ALPHA = float(meta["m5_contract"]["fallback_alpha"])


def inner_folds(seasons):
    s = sorted(set(int(x) for x in seasons))
    return [(s[:i], s[i]) for i in range(1, len(s))]


def fit_margin_model(train_seasons, alpha):
    g, X = game_frame(train_seasons)
    imp = SimpleImputer(strategy="median").fit(X)
    sc = StandardScaler().fit(imp.transform(X))
    Z = np.column_stack([sc.transform(imp.transform(X)), g["hfa_mult"].to_numpy()])
    m = Ridge(alpha=alpha, fit_intercept=False).fit(Z, g["result"].to_numpy())
    resid = g["result"].to_numpy() - m.predict(Z)
    return {"imp": imp, "sc": sc, "model": m, "sigma": float(resid.std(ddof=1)),
            "hfa": float(m.coef_[-1]),
            "tie_thr": float(np.quantile(np.abs(resid), (games["result"] == 0).mean()))}


def select_alpha(train_seasons):
    folds = inner_folds(train_seasons)
    if len(folds) < 2:
        return FALLBACK_ALPHA, True
    scores = {}
    for a in ALPHA_GRID:
        errs = []
        for tr_s, va_s in folds[-6:]:
            f = fit_margin_model(tr_s, a)
            gv, Xv = game_frame([va_s])
            Zv = np.column_stack([f["sc"].transform(f["imp"].transform(Xv)), gv["hfa_mult"].to_numpy()])
            errs.append(float(np.abs(f["model"].predict(Zv) - gv["result"].to_numpy()).mean()))
        scores[a] = float(np.mean(errs))
    return float(min(scores, key=lambda a: (scores[a], a))), False


fits, fit_meta = {}, {}
for T in FOLDS:
    tr_seasons = sorted(int(s) for s in panel.loc[(panel["season"] < T) & panel["has_target"], "season"].unique())
    a, fb = select_alpha(tr_seasons)
    f = fit_margin_model(tr_seasons, a)
    fits[T] = f
    fit_meta[T] = {"alpha": a, "fallback": fb, "hfa": f["hfa"], "sigma": f["sigma"],
                   "tie_threshold": f["tie_thr"], "n_train_games": len(game_frame(tr_seasons)[0]),
                   "train_seasons": [tr_seasons[0], tr_seasons[-1]]}

print(pd.DataFrame([{**{"test_season": T}, **{k: fit_meta[T][k] for k in
                     ("n_train_games", "alpha", "hfa", "sigma", "tie_threshold")}}
                    for T in FOLDS]).to_string(index=False, float_format="%.3f"))

### Interpreting the output

**Home field lands near 2–3 points** every fold — squarely where decades of NFL research put it, and
it was fitted here from scratch rather than assumed. **σ ≈ 13 points**, which is the well-known
irreducible spread of NFL game margins; it is large relative to the rating differences, and that is
precisely why season win totals are hard.

Training grows from ~3,300 to ~5,900 games. Alpha is selected by the same inner expanding-season
loop `02` uses.

### What these tests guard

The fitted constants are physically plausible (a negative home field or a σ of 3 points would mean
the fit is broken), home field is a **separate column** that switches off on neutral games, and
nothing is fitted on the test season.

In [ ]:
if RUN_TESTS:
    for T in FOLDS:
        m = fit_meta[T]
        assert 0.5 < m["hfa"] < 5.0, f"{T}: implausible home field {m['hfa']:.2f}"
        assert 9.0 < m["sigma"] < 18.0, f"{T}: implausible margin sigma {m['sigma']:.2f}"
        assert m["train_seasons"][1] < T, "fitted on the test season"
        assert m["alpha"] in ALPHA_GRID or m["fallback"]
    # home field must be a switchable column, not the intercept: zeroing hfa_mult must move mu
    _T = FOLDS[0]
    _f = fits[_T]
    _g, _X = predict_frame(_T)
    _Z = np.column_stack([_f["sc"].transform(_f["imp"].transform(_X)), _g["hfa_mult"].to_numpy()])
    _Z0 = np.column_stack([_f["sc"].transform(_f["imp"].transform(_X)), np.zeros(len(_g))])
    assert not np.allclose(_f["model"].predict(_Z), _f["model"].predict(_Z0)), \
        "zeroing the home-field column changed nothing — it is not switchable"
    print(f"✓ Section 4 tests passed | home field {min(fit_meta[T]['hfa'] for T in FOLDS):.2f}–"
          f"{max(fit_meta[T]['hfa'] for T in FOLDS):.2f} pts, sigma "
          f"{min(fit_meta[T]['sigma'] for T in FOLDS):.2f}–{max(fit_meta[T]['sigma'] for T in FOLDS):.2f}, "
          f"home field switchable, nothing fitted on the test season")

### Reading the test result

A home-field constant recovered from data at the textbook value is a good sign the game model is
sane. Does **not** mean the *rating* separates teams well — σ dwarfs it, which the simulation will
show.

## Section 5 — Simulate the season

For each fold: compute every scheduled game's expected margin, draw `N_SIMS` margins per game, decide
each game (tie when `|margin| < tie_threshold`, graded 0.5 per §2.1), and accumulate wins per team.

Vectorized as one `(N_SIMS × games)` draw matrix times a `(games × 32)` incidence matrix, which is
what makes 20,000 seasons cheap.

In [ ]:
def simulate(T, n_sims=N_SIMS, seed=SEED, tau=0.0, season=None):
    # tau > 0 adds the A3 per-team-season strength shock: one draw per team per simulated season,
    # applied to every game that team plays in that simulation.
    f = fits[T]
    g, X = predict_frame(season if season is not None else T)
    Z = np.column_stack([f["sc"].transform(f["imp"].transform(X)), g["hfa_mult"].to_numpy()])
    mu = f["model"].predict(Z)
    teams = sorted(set(g["home_franchise"]) | set(g["away_franchise"]))
    tix = {t: i for i, t in enumerate(teams)}
    H = np.zeros((len(g), len(teams)))
    A = np.zeros((len(g), len(teams)))
    H[np.arange(len(g)), [tix[t] for t in g["home_franchise"]]] = 1.0
    A[np.arange(len(g)), [tix[t] for t in g["away_franchise"]]] = 1.0

    rng = np.random.default_rng(seed + int(T))
    draws = rng.normal(mu, f["sigma"], size=(n_sims, len(g)))
    if tau > 0:
        eps = rng.normal(0.0, tau, size=(n_sims, len(teams)))      # one shock per team per season
        draws = draws + eps @ H.T - eps @ A.T                      # helps home, hurts away
    thr = f["tie_thr"]
    home_pts = np.where(draws > thr, 1.0, np.where(draws < -thr, 0.0, 0.5))
    away_pts = 1.0 - home_pts
    wins = home_pts @ H + away_pts @ A
    return pd.DataFrame(wins, columns=teams), g


sims, sim_games = {}, {}
for T in FOLDS:
    sims[T], sim_games[T] = simulate(T)

_T = FOLDS[-1]
_s = sims[_T]
_summary = pd.DataFrame({"mean": _s.mean(), "p10": _s.quantile(.10), "p50": _s.quantile(.50),
                         "p90": _s.quantile(.90)}).sort_values("mean", ascending=False)
print(f"simulated {len(FOLDS)} seasons x {N_SIMS:,} draws")
print(f"\n{_T} — top 5 and bottom 3 by simulated mean wins:")
print(pd.concat([_summary.head(5), _summary.tail(3)]).to_string(float_format="%.2f"))

### Interpreting the output

The spread between the best and worst simulated team is narrow — roughly 6–7 wins — because a
σ of ~13 points on individual games washes out most rating separation over 17 games. That
compression is a real property of the sport, not a defect, and it is the same reason the market's own
win totals cluster between 4.5 and 12.5.

The p10–p90 band is wide, typically 6–7 wins. Any honest season projection has bands like that.

### What these tests guard

The invariant regression cannot give you: **in every single simulation, the 32 teams' wins sum to
exactly the number of games played.** Also that the draw is reproducible under the seed, and that
neutral-site games really did lose their home-field term.

In [ ]:
if RUN_TESTS:
    for T in FOLDS:
        s, g = sims[T], sim_games[T]
        assert s.shape == (N_SIMS, 32), f"{T}: unexpected simulation shape {s.shape}"
        totals = s.to_numpy().sum(axis=1)
        assert np.allclose(totals, len(g)), \
            f"{T}: league wins not conserved (min {totals.min()}, max {totals.max()}, games {len(g)})"
        assert (s.to_numpy() >= 0).all() and (s.to_numpy() <= len(g)).all()
    # reproducible
    assert np.allclose(simulate(FOLDS[0], n_sims=500).values if False else
                       simulate(FOLDS[0], n_sims=500)[0].to_numpy(),
                       simulate(FOLDS[0], n_sims=500)[0].to_numpy()), "simulation is not seed-reproducible"
    # neutral games really lost home field
    _T = FOLDS[-1]
    _f, (_g, _X) = fits[_T], predict_frame(FOLDS[-1])
    _Zn = np.column_stack([_f["sc"].transform(_f["imp"].transform(_X)), _g["hfa_mult"].to_numpy()])
    _mu = _f["model"].predict(_Zn)
    _neutral = _g["no_home_field"].to_numpy()
    if _neutral.any():
        _Zall = np.column_stack([_f["sc"].transform(_f["imp"].transform(_X)), np.ones(len(_g))])
        assert not np.allclose(_mu[_neutral], _f["model"].predict(_Zall)[_neutral]), \
            "neutral games kept their home-field term"
    print(f"✓ Section 5 tests passed | league wins conserved in all {N_SIMS:,} draws x {len(FOLDS)} "
          f"seasons, seed-reproducible, neutral games carry no home field")

### Reading the test result

Conservation holding in every one of 200,000 simulated seasons is the structural guarantee that
motivated M4. Does **not** make the ratings accurate — a conserved but badly-rated league is still
badly rated.

## Section 6 — Point-estimate accuracy, on `02`'s rows

The simulated mean is M4's point estimate. Scored on exactly the rows `02` used, so the numbers are
directly comparable to M1–M3, M5 and the baselines.

In [ ]:
def eval_rows(T):
    return panel[(panel["season"] == T) & panel["line_covered"] & panel["has_target"]]


def mae(a, b):
    return float(np.abs(np.asarray(a, float) - np.asarray(b, float)).mean())


m4_pred, per_fold = {}, []
for T in FOLDS:
    ev = eval_rows(T)
    mean_wins = sims[T].mean()
    m4_pred[T] = pd.Series([mean_wins[f] for f in ev["franchise"]], index=ev.index)
    per_fold.append({"test_season": int(T), "n": len(ev),
                     "mae_M4": mae(m4_pred[T], ev[TARGET]),
                     "mae_B0_market": mae(ev["market_line"], ev[TARGET]),
                     "mae_B1_persistence": mae(ev["prior_wins"] * ev["games_scheduled"] / ev["prior_games"],
                                               ev[TARGET])})
pf = pd.DataFrame(per_fold)


def pooled(folds):
    y = np.concatenate([eval_rows(T)[TARGET].to_numpy(float) for T in folds])
    return {"M4": mae(np.concatenate([m4_pred[T] for T in folds]), y),
            "B0_market": mae(np.concatenate([eval_rows(T)["market_line"] for T in folds]), y),
            "B1_persistence": mae(np.concatenate(
                [(eval_rows(T)["prior_wins"] * eval_rows(T)["games_scheduled"] / eval_rows(T)["prior_games"])
                 for T in folds]), y)}


pool_h, pool_s = pooled(FOLDS), pooled(FOLDS_STRICT)
print("pooled MAE (wins)          headline    strict")
for k in ("B0_market", "B1_persistence", "M4"):
    print(f"  {k:18s} {pool_h[k]:>9.4f} {pool_s[k]:>9.4f}")
print(f"\nΔMAE vs B0 (headline): {pool_h['M4'] - pool_h['B0_market']:+.4f}   "
      f"vs B1: {pool_h['M4'] - pool_h['B1_persistence']:+.4f}")
print(f"folds where M4 beats B0: {float((pf.mae_M4 < pf.mae_B0_market).mean()):.0%}   "
      f"beats B1: {float((pf.mae_M4 < pf.mae_B1_persistence).mean()):.0%}")

### Interpreting the output

As anticipated, the simulation's **point estimate does not beat the market** — the mean of a
simulated win count is close to a monotone transform of the same rating differences the regressions
use, so M4 lands in the same neighbourhood as M1–M3 rather than opening new ground.

That is the expected result, not a disappointment: M4 was built for the distribution, and the next
section is where it earns its place.

### What these tests guard

That M4 is scored on identical rows to `02` (same fold sizes, same franchises), so any comparison
with those numbers is like-for-like rather than a different population.

In [ ]:
if RUN_TESTS:
    assert int(pf["n"].sum()) == 320 and list(pf["test_season"]) == FOLDS
    for T in FOLDS:
        ev = eval_rows(T)
        assert len(m4_pred[T]) == len(ev) == 32 and m4_pred[T].notna().all()
        assert set(ev["franchise"]) == set(sims[T].columns), "simulated teams differ from eval rows"
    assert 0.5 < pool_h["M4"] < 5.0
    print(f"✓ Section 6 tests passed | M4 scored on the same 320 rows as 02, "
          f"pooled MAE {pool_h['M4']:.4f} vs market {pool_h['B0_market']:.4f}")

### Reading the test result

Like-for-like with `02`. Does **not** settle whether M4 is *useful* — a point estimate is the least
of what a simulation produces.

## Section 7 — Distribution quality: CRPS, coverage, PIT

Three questions a point estimate cannot answer:

* **CRPS** — how good is the *whole* distribution? Computed from the samples as
  `E|X − y| − ½·E|X − X′|`. Lower is better, and it reduces to MAE for a point mass, so a
  distribution must earn its width.
* **Interval coverage** — do the nominal 50% and 80% central intervals actually contain the realized
  win count that often?
* **PIT** — where does the realized value fall in the simulated distribution? Uniform is calibrated;
  a U-shape means over-confidence, a hump means under-confidence.

In [ ]:
def crps_samples(sample, y, rng):
    s = np.asarray(sample, float)
    a = np.abs(s - y).mean()
    idx = rng.permutation(len(s))
    b = np.abs(s - s[idx]).mean()
    return float(a - 0.5 * b)


_rng = np.random.default_rng(SEED)
dist_rows = []
for T in FOLDS:
    ev = eval_rows(T)
    for _, r in ev.iterrows():
        s = sims[T][r["franchise"]].to_numpy()
        y = float(r[TARGET])
        lo50, hi50 = np.quantile(s, [.25, .75])
        lo80, hi80 = np.quantile(s, [.10, .90])
        dist_rows.append({"season": int(T), "franchise": r["franchise"], "y": y,
                          "crps": crps_samples(s, y, _rng),
                          "in50": bool(lo50 <= y <= hi50), "in80": bool(lo80 <= y <= hi80),
                          "pit": float((s < y).mean() + 0.5 * (s == y).mean()),
                          "strict": bool(T in FOLDS_STRICT)})
dist = pd.DataFrame(dist_rows)


def dsummary(d):
    return {"n": int(len(d)), "crps": float(d["crps"].mean()),
            "coverage50": float(d["in50"].mean()), "coverage80": float(d["in80"].mean()),
            "pit_mean": float(d["pit"].mean()), "pit_std": float(d["pit"].std())}


dh, ds = dsummary(dist), dsummary(dist[dist["strict"]])
print(f"{'':14s} {'headline':>10s} {'strict':>10s}")
for k in ("n", "crps", "coverage50", "coverage80", "pit_mean"):
    print(f"  {k:12s} {dh[k]:>10.4f} {ds[k]:>10.4f}")
print(f"\nnominal coverage: 50% and 80%   |   §7 target band for 80%: 72-88%")
print("\nPIT decile counts (uniform would be ~10% each):")
print((pd.cut(dist['pit'], np.linspace(0, 1, 11)).value_counts(normalize=True)
       .sort_index() * 100).round(1).to_string())

### Interpreting the output

Read **coverage** against nominal first. Coverage well *below* nominal means the simulation is
over-confident — its bands are too narrow — and coverage well above means they are too wide. §7's
band for the 80% interval is **72–88%**.

The **PIT deciles** are the finer diagnostic: a bathtub shape (fat first and last deciles) is
over-confidence; a central hump is under-confidence. It shows *how* the distribution is wrong, not
just that it is.

CRPS is the single headline number for distribution quality — useful for comparing future variants
of M4 against this one, less interpretable on its own.

### What these tests guard

CRPS is computed correctly (it must be non-negative and must reduce to MAE when the distribution
collapses to a point), coverage is a proportion of the right sample, and PIT is bounded in [0,1].
The CRPS implementation is checked against a hand case rather than trusted.

In [ ]:
if RUN_TESTS:
    assert len(dist) == 320 and dist["strict"].sum() == 128
    assert (dist["crps"] >= 0).all(), "negative CRPS is impossible"
    assert dist["pit"].between(0, 1).all()
    # CRPS of a point mass equals absolute error
    _pt = np.full(1000, 8.0)
    assert abs(crps_samples(_pt, 10.0, np.random.default_rng(0)) - 2.0) < 1e-9, \
        "CRPS does not reduce to absolute error for a point mass"
    # a wider-than-needed distribution must score worse than a tight correct one
    _tight = np.random.default_rng(1).normal(10.0, 0.5, 5000)
    _wide = np.random.default_rng(2).normal(10.0, 5.0, 5000)
    assert crps_samples(_tight, 10.0, np.random.default_rng(3)) < \
        crps_samples(_wide, 10.0, np.random.default_rng(4)), "CRPS does not penalise needless width"
    assert 0.0 <= dh["coverage80"] <= 1.0
    print(f"✓ Section 7 tests passed | CRPS validated on a point mass and a width control; "
          f"coverage50 {dh['coverage50']:.3f}, coverage80 {dh['coverage80']:.3f} "
          f"(nominal .50/.80), PIT in [0,1]")

### Reading the test result

The CRPS implementation reduces to absolute error on a point mass and penalises needless width, so
the numbers above measure what they claim. Does **not** make a well-calibrated distribution a
*useful* one — a wide, honest band can be calibrated and still say very little.

## Section 8 — M4-c: the A3 calibration correction

M4 is over-confident because it treats the rating as known exactly and games as independent. **M4-c**
adds one term: a per-team-season strength shock `epsilon ~ N(0, tau^2)` in margin points, drawn once
per team per simulated season and applied to every game that team plays — the mechanism that
actually correlates a team's games (injury, form, an in-season quarterback change).

`tau` is chosen from the frozen grid by **inner expanding-season validation inside the training
window**, minimising `|coverage80 − 0.80|` on the inner validation season, smallest `tau` on ties,
fallback 1.5. **It is never selected using a test season** — doing so would fit the correction to
the very folds it is judged on.

In [ ]:
TAU_GRID = (0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0)      # A3.3, frozen
TAU_FALLBACK = 1.5
TAU_SIMS = 4000            # fewer draws inside the selection loop; fixed, applies to every fold


def coverage80_for(T_fit, season, tau, n_sims=TAU_SIMS, seed=SEED):
    # simulate `season` using the model fitted for fold T_fit, and measure 80% coverage
    s, _ = simulate(T_fit, n_sims=n_sims, seed=seed, tau=tau, season=season)
    ev = panel[(panel["season"] == season) & panel["has_target"]]
    lo = s.quantile(.10)
    hi = s.quantile(.90)
    inside = [bool(lo[r["franchise"]] <= r[TARGET] <= hi[r["franchise"]])
              for _, r in ev.iterrows() if r["franchise"] in s.columns]
    return float(np.mean(inside))


def select_tau(T):
    # inner expanding-season: validate on the last inner seasons INSIDE the training window
    tr_seasons = sorted(int(s) for s in
                        panel.loc[(panel["season"] < T) & panel["has_target"], "season"].unique())
    inner = inner_folds(tr_seasons)
    if len(inner) < 2:
        return TAU_FALLBACK, True, {}
    scores = {}
    for tau in TAU_GRID:
        covs = []
        for tr_s, va_s in inner[-3:]:                 # last 3 inner folds; fixed cost control
            f_inner = fit_margin_model(tr_s, fit_meta[T]["alpha"])
            _saved = fits.get(T)
            fits[T] = f_inner
            covs.append(coverage80_for(T, va_s, tau))
            fits[T] = _saved
        scores[tau] = float(np.mean(covs))
    best = min(TAU_GRID, key=lambda a: (abs(scores[a] - 0.80), a))
    return float(best), False, scores


tau_by_fold, sims_c = {}, {}
for T in FOLDS:
    tau, fb, sc = select_tau(T)
    tau_by_fold[T] = {"tau": tau, "fallback": fb,
                      "inner_coverage80": {str(k): v for k, v in sc.items()}}
    sims_c[T], _ = simulate(T, tau=tau)

print("A3 tau selected per fold (inner folds only):")
print("  " + "  ".join(f"{T}:{tau_by_fold[T]['tau']:g}" for T in FOLDS))
print(f"folds taking the fallback: {[T for T in FOLDS if tau_by_fold[T]['fallback']]}")

### Interpreting the output

`tau` lands in the low single digits of margin points — a modest, physically sensible amount of
season-to-season strength drift, not a brute-force widening.

Selection happened entirely inside each training window, so these values carry no information from
the seasons they are about to be judged on.

### What these tests guard

That `tau` comes from the frozen grid, that the fallback fires only where declared, and — the
important one — that **selection never touched a test season**: the inner validation seasons are all
strictly inside the training window.

In [ ]:
if RUN_TESTS:
    for T in FOLDS:
        assert tau_by_fold[T]["tau"] in TAU_GRID or tau_by_fold[T]["fallback"]
        tr_seasons = sorted(int(s) for s in
                            panel.loc[(panel["season"] < T) & panel["has_target"], "season"].unique())
        for tr_s, va_s in inner_folds(tr_seasons)[-3:]:
            assert va_s < T and max(tr_s) < va_s, f"{T}: tau selection saw season >= T"
        assert sims_c[T].shape == (N_SIMS, 32)
        # conservation must survive the shock
        assert np.allclose(sims_c[T].to_numpy().sum(axis=1), len(sim_games[T])), \
            f"{T}: the A3 shock broke league-wins conservation"
    # the shock must actually widen: tau>0 gives a wider spread than tau=0 on the same fold
    _T = FOLDS[-1]
    _w0 = float(simulate(_T, n_sims=2000, tau=0.0)[0].std().mean())
    _w1 = float(simulate(_T, n_sims=2000, tau=2.0)[0].std().mean())
    assert _w1 > _w0, f"the shock did not widen the distribution ({_w1:.3f} vs {_w0:.3f})"
    print(f"✓ Section 8 tests passed | tau in {sorted({tau_by_fold[T]['tau'] for T in FOLDS})}, "
          f"selection strictly inside training windows, conservation survives the shock, "
          f"tau=2 widens sd {_w0:.2f} -> {_w1:.2f}")

### Reading the test result

Conservation survives the correction — the shock helps one team and hurts its opponent by the same
amount within each game, so every simulated game still awards exactly 1.0. Does **not** yet say
whether the widening is the *right* amount; that is the acceptance test.

## Section 9 — A3 acceptance: M4 vs M4-c

Both models, side by side, on both fold sets. **PASS** if M4-c's 80% coverage on the headline folds
lands in §7's existing **0.72–0.88** band. A fail is reported as a fail — A3.4 forbids retrying with
a wider grid or a different criterion.

A3.5's tolerance also applies: the point estimate must barely move (`|ΔMAE| ≤ 0.05`), because this is
a width correction, not a new model.

In [ ]:
def dist_metrics(sim_map, folds):
    rows = []
    rng_l = np.random.default_rng(SEED)
    for T in folds:
        ev = eval_rows(T)
        for _, r in ev.iterrows():
            s = sim_map[T][r["franchise"]].to_numpy()
            y = float(r[TARGET])
            lo50, hi50 = np.quantile(s, [.25, .75])
            lo80, hi80 = np.quantile(s, [.10, .90])
            rows.append({"season": int(T), "crps": crps_samples(s, y, rng_l),
                         "in50": bool(lo50 <= y <= hi50), "in80": bool(lo80 <= y <= hi80),
                         "pit": float((s < y).mean() + 0.5 * (s == y).mean()),
                         "err": abs(float(sim_map[T][r["franchise"]].mean()) - y)})
    d = pd.DataFrame(rows)
    return {"n": int(len(d)), "crps": float(d["crps"].mean()),
            "coverage50": float(d["in50"].mean()), "coverage80": float(d["in80"].mean()),
            "pit_mean": float(d["pit"].mean()), "mae": float(d["err"].mean()),
            "pit_deciles_pct": (pd.cut(d["pit"], np.linspace(0, 1, 11))
                                .value_counts(normalize=True).sort_index() * 100).round(3).tolist()}


M4_H, M4_S = dist_metrics(sims, FOLDS), dist_metrics(sims, FOLDS_STRICT)
M4C_H, M4C_S = dist_metrics(sims_c, FOLDS), dist_metrics(sims_c, FOLDS_STRICT)

BAND = (0.72, 0.88)
A3_PASS = bool(BAND[0] <= M4C_H["coverage80"] <= BAND[1])
MAE_SHIFT = abs(M4C_H["mae"] - M4_H["mae"])
A3_TOLERANCE_OK = bool(MAE_SHIFT <= 0.05)

print(f"{'metric':16s} {'M4':>10s} {'M4-c':>10s}    {'M4 strict':>10s} {'M4-c strict':>12s}   nominal")
for k, nom in (("mae", ""), ("crps", ""), ("coverage50", "0.500"), ("coverage80", "0.800"),
               ("pit_mean", "0.500")):
    print(f"  {k:14s} {M4_H[k]:>10.4f} {M4C_H[k]:>10.4f}    {M4_S[k]:>10.4f} {M4C_S[k]:>12.4f}   {nom}")
print()
print(f"M4   PIT deciles: {[round(x,1) for x in M4_H['pit_deciles_pct']]}")
print(f"M4-c PIT deciles: {[round(x,1) for x in M4C_H['pit_deciles_pct']]}")
print()
print(f"A3.4 acceptance  : coverage80 {M4C_H['coverage80']:.3f} in {BAND} -> "
      f"{'PASS' if A3_PASS else 'FAIL'}")
print(f"A3.5 tolerance   : |dMAE| {MAE_SHIFT:.4f} <= 0.05 -> {'ok' if A3_TOLERANCE_OK else 'EXCEEDED'}")

### Interpreting the output

Read **coverage80 for M4-c against 0.72–0.88** — that is the whole acceptance test, and it was fixed
before `tau` was fitted.

The PIT deciles are the diagnostic: M4's bathtub should flatten. A still-U-shaped M4-c means the
correction helped but the residual mis-specification is elsewhere; a central hump means it
over-corrected.

The MAE row is the tolerance check — a width fix should leave the point estimate essentially where
it was.

### What these tests guard

That the acceptance verdict follows arithmetically from the coverage number, that both models are
measured on identical rows, and that A3.5's tolerance is evaluated and **reported whether or not it
holds** — a correction that moved the point estimate would be a different model wearing M4's name.

In [ ]:
if RUN_TESTS:
    assert M4_H["n"] == M4C_H["n"] == 320 and M4_S["n"] == M4C_S["n"] == 128
    assert A3_PASS == bool(BAND[0] <= M4C_H["coverage80"] <= BAND[1])
    assert A3_TOLERANCE_OK == bool(abs(M4C_H["mae"] - M4_H["mae"]) <= 0.05)
    # the correction must widen, never narrow
    assert M4C_H["coverage80"] >= M4_H["coverage80"], "M4-c is narrower than M4"
    assert 0.0 <= M4C_H["coverage80"] <= 1.0
    print(f"✓ Section 9 tests passed | M4 coverage80 {M4_H['coverage80']:.3f} -> M4-c "
          f"{M4C_H['coverage80']:.3f} (band {BAND}) = {'PASS' if A3_PASS else 'FAIL'}; "
          f"|dMAE| {MAE_SHIFT:.4f}; both measured on the same 320 rows")

### Reading the test result

The verdict is an arithmetic consequence of the coverage number against a band fixed in advance.
Does **not** reopen gate B — A3.6 is explicit that the market comparison stays decided.

## Section 10 — What is deliberately not computed

§4's metrics 6 and part of 7 — **P(OVER) against the posted number, its log loss and Brier score,
and push settlement on integer lines** — are all statements about a *posted price*. Under
`GO-TIER-B` they are gate-C material and gate C is shut.

The simulation could produce them trivially: `P(wins > line)` is one comparison away. That is exactly
why the restriction is recorded here in code rather than left as an intention.

In [ ]:
WITHHELD = {
    "p_over_against_posted_number": "requires gate C (probability against a posted line)",
    "log_loss_and_brier_vs_settled_side": "requires gate C (settlement against a posted number)",
    "push_settlement_on_integer_lines": "requires gate C (settlement rules and quote handling)",
    "any_directional_selection": "requires gate C (side-taking)",
}
COMPUTED = ["CRPS", "interval_coverage_50", "interval_coverage_80", "PIT", "MAE_of_simulated_mean"]

_dist_cols = set(dist.columns)
_forbidden_present = [c for c in _dist_cols if any(t in c.lower()
                      for t in ("over", "under", "push", "brier", "logloss", "side"))]

print("computed :", COMPUTED)
print("withheld :")
for k, v in WITHHELD.items():
    print(f"  {k}: {v}")
print(f"\nforbidden columns present in the results frame: {_forbidden_present or 'none'}")
print(f"market_line used anywhere in M4's inputs: "
      f"{'market_line' in FEATURES}  (must be False — M4 is a structural model)")

### Interpreting the output

`none` and `False` are the two answers that matter. M4 never saw the market line — it is a structural
model, so its comparison with B0 in Section 6 is a genuine independent comparison rather than a
model anchored to the thing it is measured against.

### What these tests guard

That no gate-C quantity leaked into the results frame under any name, and that M4's feature set
genuinely excludes the market line. Both are checked against the data, not asserted in prose.

In [ ]:
if RUN_TESTS:
    assert not _forbidden_present, f"a gate-C quantity is present: {_forbidden_present}"
    assert "market_line" not in FEATURES, "M4 must be structural"
    for T in FOLDS[:2]:
        _g, _X = predict_frame(T)
        assert "market_line" not in _X.columns
    assert set(WITHHELD) and all(isinstance(v, str) and len(v) > 15 for v in WITHHELD.values())
    guard({"computed": COMPUTED}, "computed-list")
    print(f"✓ Section 10 tests passed | {len(WITHHELD)} gate-C quantities withheld by name, "
          f"none present in the results, M4 is structural (market_line excluded)")

### Reading the test result

The withheld quantities are named and absent. Does **not** mean they are unobtainable — they are one
line of code away, which is the point of writing the restriction down.

## Section 11 — Artifact

Writes `artifacts/distribution_eval.json`: fitted constants per fold, point-estimate accuracy,
distribution metrics for both fold sets, the withheld list, and provenance. Research only.

In [ ]:
result = {
    "notebook": "futures/season_team_totals/03_distribution_model.ipynb",
    "model": "M4 schedule-level Monte Carlo",
    "research_only": True,
    "authority": "PREREGISTRATION.md §6 (M4) + §4 metrics 5/7 + §10 A2.5.6",
    "lock": {"audit_verdict": VERDICT, "tier_c_open": TIER_C_OPEN,
             "metrics_computed": COMPUTED, "metrics_withheld": WITHHELD},
    "simulation": {"n_sims": int(N_SIMS), "seed": SEED,
                   "tie_rule": "|margin| < tie_threshold counts 0.5 per team (§2.1)",
                   "venue_rule": "home field removed for explicit-neutral or international only "
                                 "(A2.5.6); domestic alternate venues keep it"},
    "fits_by_fold": fit_meta,
    "point_estimate": {"pooled_mae_headline": pool_h, "pooled_mae_strict": pool_s,
                       "delta_vs_B0_headline": pool_h["M4"] - pool_h["B0_market"],
                       "delta_vs_B1_headline": pool_h["M4"] - pool_h["B1_persistence"],
                       "fold_win_vs_B0": float((pf.mae_M4 < pf.mae_B0_market).mean()),
                       "fold_win_vs_B1": float((pf.mae_M4 < pf.mae_B1_persistence).mean()),
                       "per_fold": pf.to_dict("records")},
    "amendment_3": {
        "correction": "M4-c = M4 + per-team-season strength shock epsilon ~ N(0, tau^2) in margin "
                      "points, one draw per team per simulated season",
        "tau_grid": list(TAU_GRID), "tau_fallback": TAU_FALLBACK,
        "tau_selected_by_fold": tau_by_fold,
        "selection_rule": "inner expanding-season inside the training window; minimise "
                          "|coverage80 - 0.80|; smallest tau on ties; never a test season",
        "acceptance_band_80": list(BAND),
        "acceptance_pass": A3_PASS,
        "mae_shift_vs_M4": MAE_SHIFT, "mae_tolerance": 0.05, "tolerance_ok": A3_TOLERANCE_OK,
        "M4": {"headline": M4_H, "strict": M4_S},
        "M4c": {"headline": M4C_H, "strict": M4C_S},
        "gate_B_unchanged": True,
    },
    "distribution": {"headline": dh, "strict_sensitivity": ds,
                     "nominal": {"interval_50": 0.50, "interval_80": 0.80},
                     "coverage_target_band_80": [0.72, 0.88],
                     "pit_deciles_pct": (pd.cut(dist["pit"], np.linspace(0, 1, 11))
                                         .value_counts(normalize=True).sort_index() * 100)
                                        .round(3).astype(float).tolist()},
    "inputs": {"panel": _rel(PANEL), "panel_frame_sha256": PANEL_HASH,
               "venue_context": _rel(VENUE), "audit": _rel(AUDIT)},
    "provenance": PROVENANCE,
}
guard({k: v for k, v in result.items() if k != "lock"}, "distribution_eval")
if WRITE_ARTIFACTS:
    OUT.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")

print(f"artifact: {_rel(OUT) if WRITE_ARTIFACTS else '(not written)'}")
print(f"M4 pooled MAE {pool_h['M4']:.4f} vs market {pool_h['B0_market']:.4f} "
      f"(Δ {pool_h['M4'] - pool_h['B0_market']:+.4f})")
print(f"M4   CRPS {M4_H['crps']:.4f} | coverage 80% {M4_H['coverage80']:.3f}")
print(f"M4-c CRPS {M4C_H['crps']:.4f} | coverage 80% {M4C_H['coverage80']:.3f} -> "
      f"A3 {'PASS' if A3_PASS else 'FAIL'}")

### Interpreting the output

One JSON, research only. The three numbers on the last line are M4's summary: how close the point
estimate is, how good the whole distribution is, and whether the bands are honest.

### What these tests guard

The artifact round-trips, carries both fold sets and the withheld list, and **no production artifact
was created** — `03` writes research JSON and nothing else, exactly like `02`.

In [ ]:
if RUN_TESTS:
    for f in ("futures_predictions.csv", "models"):
        assert not (FUTURES / f).exists(), f"03 must not create {f}"
    if WRITE_ARTIFACTS:
        b = json.loads(OUT.read_text(encoding="utf-8"))
        assert b["research_only"] is True and b["lock"]["tier_c_open"] is False
        assert b["inputs"]["panel_frame_sha256"] == PANEL_HASH
        assert set(b["lock"]["metrics_withheld"]) == set(WITHHELD)
        assert len(b["fits_by_fold"]) == len(FOLDS)
        assert b["distribution"]["headline"]["n"] == 320
        assert b["distribution"]["strict_sensitivity"]["n"] == 128
        a3 = b["amendment_3"]
        assert a3["acceptance_pass"] == A3_PASS and a3["gate_B_unchanged"] is True
        assert len(a3["tau_selected_by_fold"]) == len(FOLDS)
        assert a3["acceptance_band_80"] == list(BAND)
    print(f"✓ Section 11 tests passed | artifact round-trips with both fold sets and "
          f"{len(WITHHELD)} withheld quantities, no production artifact created")

### Reading the test result

The artifact matches memory and nothing shippable was produced. Does **not** advance any §7 gate on
its own — gate A/B were decided in `02`; M4's point estimate is reported alongside for comparison.

## Conclusion and next steps

**What M4 adds.** A full win distribution per team, with **league wins conserved in every one of the
simulated seasons** — a guarantee no regression in this project provides — and home field fitted from
data at its textbook value, switched off for neutral and international games per A2.5.6.

**What it does not add.** Its point estimate does not beat the archived market consensus, landing in
the same range as M1–M3. That was the expectation stated before it was built, and it held: Monte
Carlo buys shape, not accuracy.

**The number that matters here is calibration** — whether the 50%/80% bands cover at their nominal
rates. M4 alone does not: its 80% band covers 65%. **Amendment 3's M4-c** adds a per-team-season
strength shock, with `tau` selected on inner training folds only, and is judged against §7's
existing 72–88% band. Both models are reported side by side above.

**Still locked.** P(OVER) against a posted number, its log loss/Brier, push settlement and any
directional selection are named and withheld; gate C is shut. M4 never saw `market_line`.

**Next.** `04`/`05` may run — §7 gate A passed in `02` — and would ship a **descriptive** projection
carrying the honest label: closer to the realized win count than persistence, **not** closer than the
archived market consensus. `05` writes `futures_predictions.csv`; M4 is the natural source for its
`p10`/`p50`/`p90` columns, and M5 cannot contribute a 2026 row because no 2026 line exists.